# System: coordinar Agents · 2.1.4

La gramática del grafo es la misma. Los nodos ahora son Agents. LangGraph sigue siendo el único motor.

In [2]:
import pydantic
import agentic_systems as toolkit
assert toolkit.__version__ == '2.1.4'

class State(pydantic.BaseModel):
    value: int
    increased: int | None = None
    doubled: int | None = None

def increase(state):
    return {'value': state.value + 1}

def double(state):
    return {'value': state.value * 2}


In [3]:
increase_graph = toolkit.Graph(State)
increase_graph.add_node('increase', increase)
increase_graph.add_edge(toolkit.START, 'increase')
increase_graph.add_edge('increase', toolkit.END)
incrementer = toolkit.Agent('incrementer', graph=increase_graph)
double_graph = toolkit.Graph(State)
double_graph.add_node('double', double)
double_graph.add_edge(toolkit.START, 'double')
double_graph.add_edge('double', toolkit.END)
multiplier = toolkit.Agent('multiplier', graph=double_graph)
graph = toolkit.Graph(State)
graph.add_node('incrementer', incrementer)
graph.add_node('multiplier', multiplier)
graph.add_edge(toolkit.START, 'incrementer')
graph.add_edge('incrementer', 'multiplier')
graph.add_edge('multiplier', toolkit.END)
system = toolkit.System('team', graph=graph)
result = system.run({'value': 2})
assert result.ok and result.data['value'] == 6
assert result.check_invariants().ok
print(result.human_results())


team: COMPLETED
Execution: run-6fbcf1f43e2c44598ba647c2e831162b
Input: {"value":2}
Output: {"value":6}
Process:
- Agent incrementer: completed
- Step increase: completed
- Agent multiplier: completed
- Step double: completed


In [4]:
def left(state):
    return {'increased': state.value + 1}

def right(state):
    return {'doubled': state.value * 2}

def total(state):
    return {'value': state.increased + state.doubled}


In [5]:
left_graph = toolkit.Graph(State)
left_graph.add_node('left', left)
left_graph.add_edge(toolkit.START, 'left')
left_graph.add_edge('left', toolkit.END)
left_agent = toolkit.Agent('left', graph=left_graph)
right_graph = toolkit.Graph(State)
right_graph.add_node('right', right)
right_graph.add_edge(toolkit.START, 'right')
right_graph.add_edge('right', toolkit.END)
right_agent = toolkit.Agent('right', graph=right_graph)
graph = toolkit.Graph(State)
graph.add_node('left', left_agent)
graph.add_node('right', right_agent)
graph.add_node('total', total)
graph.add_edge(toolkit.START, 'left')
graph.add_edge(toolkit.START, 'right')
graph.add_edge(['left', 'right'], 'total')
graph.add_edge('total', toolkit.END)
system = toolkit.System('concurrent-team', graph=graph)
result = system.run({'value': 2})
assert result.ok and result.data['value'] == 7
assert result.check_invariants().ok
print(result.human_results())


concurrent-team: COMPLETED
Execution: run-89c1c523887546e7813d88be98b071f0
Input: {"value":2}
Output: {"value":7,"increased":3,"doubled":4}
Process:
- Agent left: completed
- Step left: completed
- Agent right: completed
- Step right: completed
- Step total: completed


In [6]:
graph = toolkit.Graph(State)
graph.add_node('increase', incrementer)
graph.add_node('double', multiplier)
graph.add_edge(toolkit.START, 'increase')
graph.add_conditional_edges('increase', lambda state: 'double' if state.value >= 0 else toolkit.END, ['double', toolkit.END])
graph.add_edge('double', toolkit.END)
system = toolkit.System('decision', graph=graph)
result = system.run({'value': 2})
assert result.ok and result.data['value'] == 6
assert result.check_invariants().ok
print(result.human_results())


decision: COMPLETED
Execution: run-6adb4b369f9847cfa7a8d643020fcec6
Input: {"value":2}
Output: {"value":6}
Process:
- Agent incrementer: completed
- Step increase: completed
- Decision increase -> "double"
- Agent multiplier: completed
- Step double: completed
